In [1]:
import scanpy as sc
import muon as mu
import pandas as pd

/Users/sagniknandy/miniconda3/envs/AMP_all_projects/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Load multi-omics data

# --- Read counts ---
adata_rna_counts  = sc.read_h5ad("cleaned_rna_counts_tea_seq.h5ad")
adata_atac_counts = sc.read_h5ad("cleaned_atac_counts_tea_seq.h5ad")
adata_adt_counts  = pd.read_csv("cleaned_adt_counts.csv", index_col=0)

# --- Read normalized matrices ---
adata_rna_norm  = sc.read_h5ad("cleaned_rna_reads_tea_seq.h5ad")
adata_atac_norm = sc.read_h5ad("cleaned_atac_reads_tea_seq.h5ad")
adata_adt_norm  = pd.read_csv("cleaned_adt_normalized.csv", index_col=0)


In [3]:
# --- load counts and normalized matrices separately ---
# shapes all: (n_cells, n_features)
X_rna_counts = (adata_rna_counts.X).T       # raw RNA counts
X_rna_norm   = (adata_rna_norm.X).T       # log1p-normalized / HVG-filtered RNA

X_atac_counts = (adata_atac_counts.X).T
X_atac_norm   = (adata_atac_norm.X).T      # e.g. TF-IDF, LSI or gene-activity log1p

X_adt_counts = adata_adt_counts.values   # raw ADT counts
X_adt_norm   = adata_adt_norm.values       # your CLR / CPM-normalized ADT (like A_cpm or A)

cell_ids      = pd.read_csv("final_cell_barcodes_tea_seq.csv", index_col=0).index.tolist()
gene_names    = pd.read_csv("final_rna_features_tea_seq.csv", index_col=0).index.tolist()
peak_names    = pd.read_csv("final_atac_features_tea_seq.csv", index_col=0).index.tolist()
protein_names = pd.read_csv("final_adt_features_tea_seq.csv", index_col=0).index.tolist()
labels        = pd.read_csv("cleaned_cell_labels_meta_tea_seq.csv", index_col=1).index.tolist()      # cell type labels, length n_cells


In [4]:
# --- Merge: counts in .X, normalized in .layers ---
adata_rna  = sc.AnnData(X_rna_counts)
adata_atac = sc.AnnData(X_atac_counts)
adata_adt  = sc.AnnData(X_adt_counts)

adata_rna.layers["norm"] = X_rna_norm.copy()
adata_atac.layers["norm"]   = X_atac_norm.copy()
adata_adt.layers["norm"]     = X_adt_norm.copy()

adata_rna.obs["cell_id"] = cell_ids
adata_atac.obs["cell_id"] = cell_ids
adata_adt.obs["cell_id"] = cell_ids

adata_rna.var_names = gene_names
adata_atac.var_names = peak_names
adata_adt.var_names = protein_names

# ensure same obs index
for ad in (adata_rna, adata_atac, adata_adt):
    ad.obs.set_index("cell_id", inplace=True)

mdata = mu.MuData({"rna": adata_rna, "atac": adata_atac, "adt": adata_adt})
mdata.obs["celltype"] = labels

mdata.write("../data/multi.h5mu")

/Users/sagniknandy/miniconda3/envs/AMP_all_projects/lib/python3.10/site-packages/mudata/_core/mudata.py:1598: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("var", axis=0, join_common=join_common)
/Users/sagniknandy/miniconda3/envs/AMP_all_projects/lib/python3.10/site-packages/mudata/_core/mudata.py:963: UserWarning: Cannot join columns with the same name because var_names are intersecting.
  warnings.warn(
/Users/sagniknandy/miniconda3/envs/AMP_all_projects/lib/python3.10/site-packages/mudata/_core/mudata.py:1461: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default

In [5]:
for mod in ["rna", "atac", "adt"]:
    mdata.mod[mod].obs["celltype"] = mdata.obs["celltype"]

In [6]:
mdata.mod["rna"].write("../data/rna.h5ad")
mdata.mod["atac"].write("../data/atac.h5ad")
mdata.mod["adt"].write("../data/adt.h5ad")

In [7]:
mdata.mod["adt"]

AnnData object with n_obs × n_vars = 6335 × 40
    obs: 'celltype'
    layers: 'norm'